In [0]:
from pyspark.sql import SparkSession, functions as f
#reading Hospital A departments data
df_hosa=spark.read.parquet("abfss://bronze@teresasa.dfs.core.windows.net/hosa/departments/")

# reading Hospital B departments data
df_hosb=spark.read.parquet("abfss://bronze@teresasa.dfs.core.windows.net/hosb/departments/")

#union two departments dataframes
df_merged = df_hosa.unionByName(df_hosb)

#create the dept_id column and rename deptid to src_depid
df_merged = df_merged.withColumn("SRC_Dept_id", f.col("deptid")) \
                     .withColumn("dept_id",f.concat(f.col("deptid"),f.lit('-'),f.col("datasource")))\
                     .drop("deptid")\
                     
df_merged.createOrReplaceTempView("departments")



In [0]:
%sql
select * from departments

In [0]:
%sql
CREATE TABLE IF NOT EXISTS project_dev.silver.departments
(
    Dept_id string,
    SRC_Dept_id string,
    Name string,
    datasource string,
    is_quarantined boolean
)
USING DELTA;

In [0]:
%sql
truncate table project_dev.silver.departments

In [0]:
%sql
INSERT INTO project_dev.silver.departments
SELECT
    Dept_Id,
    SRC_Dept_Id,
    Name,
    Datasource,
        CASE 
            WHEN SRC_Dept_id IS NULL OR Dept_Id IS NULL THEN TRUE
            ELSE FALSE
        END AS is_quarantined
    FROM departments
